# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR² dataset package](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL of the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access metadata object; do not treat as a dict
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Keywords: {', '.join(getattr(metadata, 'keywords', []))}")

## 2. Data Overview
Review available record sets, fields, their `@id`, and structure.

In [ ]:
# List all record sets with their @id and fields
if hasattr(metadata, 'record_sets'):
    print('Record sets found in dataset:')
    for rs in metadata.record_sets:
        print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")
        if 'fields' in rs:
            for f in rs['fields']:
                print(f"    - Field: @id: {f['@id']}, name: {f.get('name', 'N/A')}, dataType: {f.get('dataType', 'N/A')}")
else:
    # Fallback: Try loading record set info programmatically
    if hasattr(dataset, 'record_sets'):
        for rs in dataset.record_sets:
            print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")
            for f in rs.get('fields', []):
                print(f"    - Field: @id: {f['@id']}, name: {f.get('name', 'N/A')}, dataType: {f.get('dataType', 'N/A')}")
    else:
        print('No record sets found via metadata. Attempting to list records to enumerate record set IDs...')
        # Try common record set names
        for possible_rs_id in ['cr:RecordSet', 'rs1', 'recordset', 'records', 'cr:recordSet']:
            try:
                records = list(dataset.records(record_set=possible_rs_id, limit=1))
                if records:
                    print(f'Example records found in {possible_rs_id}:')
                    print(records[0])
            except Exception as e:
                continue

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Set the record set IDs manually (inspect output above to choose the correct record set ID(s))

# Common pattern for Croissant schemas: main record set is '@id': 'cr:RecordSet:MainTable' or similar.
# For this dataset, main table appears as 'cr:RecordSet:MainTable'. If unknown, try 'cr:RecordSet'.
main_record_set_id = 'cr:RecordSet:MainTable'
secondary_record_set_id = 'cr:RecordSet:ProteinsBySample'  # Example, update if available in data

# Try loading using the main_record_set_id; fallback if necessary
try:
    records = list(dataset.records(record_set=main_record_set_id))
except Exception as e:
    print(f"Failed with {main_record_set_id}, trying 'cr:RecordSet' fallback.")
    main_record_set_id = 'cr:RecordSet'
    records = list(dataset.records(record_set=main_record_set_id))

# Preview a single record
print(f"Example record from {main_record_set_id}:")
print(records[0] if len(records) > 0 else 'No records found.')

# Load all records into a DataFrame
df = pd.DataFrame(records)
print(f"Loaded records: {len(df)} rows\nColumns: {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a numeric field and a grouping field from the record fields
# For this dataset, likely numeric fields are: 'cr:Field:coverage_percent', 'cr:Field:NumPeptides', 'cr:Field:MW', etc.
# Refer to the output from section 2 or df.columns to determine the exact available @id

numeric_field_id = 'cr:Field:coverage_percent'  # '%' coverage as typically computed in proteomics
group_field_id = 'cr:Field:sample_id'           # E.g. sample, experiment, group, or other if present

# Filter for proteins with coverage_percent above threshold
threshold = 10.0
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id].astype(float) > threshold].copy()
    print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold}")

    # Normalize the numeric field
    mean_val = filtered_df[numeric_field_id].mean()
    std_val = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_val) / std_val
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by attribute if available
    if group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped average '{numeric_field_id}' by '{group_field_id}':")
        print(grouped.head())
else:
    print(f"Field {numeric_field_id} not found in columns: {df.columns.tolist()}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of the numeric field
if numeric_field_id in df.columns:
    plt.figure(figsize=(7, 4))
    df[numeric_field_id].astype(float).hist(bins=30, color='#3792cb', alpha=0.8)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
else:
    print(f"Field {numeric_field_id} not found. Histogram cannot be created.")

# Boxplot by group if possible
if (group_field_id in df.columns) and (numeric_field_id in df.columns):
    plt.figure(figsize=(10,5))
    df.boxplot(column=numeric_field_id, by=group_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.suptitle('')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² dataset provides quantitative and attribute-rich protein information from mass spectrometry of extracellular vesicles from human mast cells.
- Numeric fields such as coverage percent, number of peptides, and molecular weight allow for statistical and comparative analysis across biological samples.
- Filtering by coverage percent reveals proteins broadly detected and suitable for normalization or downstream analysis.
- Grouping by sample or experimental group (if available) can help identify patterns specific to biological conditions or technical replicates.

> Continue your research by exploring additional fields, cross-linking with peptides, or performing more advanced multivariate analyses.